In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras

# Zadanie 1 - Pobieranie danych

In [2]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

housing = fetch_california_housing()

X_train_full, X_test, y_train_full, y_test = train_test_split(housing.data, housing.target, random_state=42)
X_train, X_valid, y_train, y_valid = train_test_split(X_train_full, y_train_full, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_valid = scaler.transform(X_valid)
X_test = scaler.transform(X_test)

# Zadanie 2 -  Przeszukiwanie przestrzeni hiperparametrów przy pomocy scikit-learn

In [3]:
from scipy.stats import reciprocal

In [4]:
n_hidden_list = [0, 1, 2, 3]
n_neurons_list = list(range(1, 101))
learning_rate_list = reciprocal(3e-4, 3e-2).rvs(1000).tolist()
optimizer_list = ['adam', 'sgd', 'nesterov']

In [5]:
param_distribs = {
"model__n_hidden": n_hidden_list,
"model__n_neurons": n_neurons_list,
"model__learning_rate": learning_rate_list,
"model__optimizer": optimizer_list
}

In [6]:
def build_model(n_hidden=1, n_neurons=30, optimizer='sgd', learning_rate=3e-3):
    model = tf.keras.models.Sequential()
    model.add(keras.layers.InputLayer(shape=[8]))
    
    for i in range(n_hidden):
        model.add(keras.layers.Dense(n_neurons, activation="relu"))
        
    model.add(keras.layers.Dense(1))
    
    if optimizer == 'adam':
        optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    elif optimizer == 'sgd':
        optimizer = keras.optimizers.SGD(learning_rate=learning_rate)
    else:
        optimizer = keras.optimizers.SGD(learning_rate=learning_rate, nesterov=True)
        
    model.compile(loss="mse", optimizer=optimizer)
    
    return model

In [7]:
# model_test_1 = build_model(learning_rate=1e-6)
# model_test_2 = build_model(learning_rate=1e-5)
# model_test_3 = build_model(learning_rate=1e-4)
# model_test_4 = build_model(learning_rate=1e-3)
# model_test_5 = build_model(learning_rate=1e-2)
# model_test_6 = build_model(learning_rate=1e-1)

In [8]:
# h1 = model_test_1.fit(X_train, y_train, epochs=5,
#                  validation_data=(X_valid, y_valid))
# h2 = model_test_2.fit(X_train, y_train, epochs=5,
#                  validation_data=(X_valid, y_valid))
# h3 = model_test_3.fit(X_train, y_train, epochs=5,
#                  validation_data=(X_valid, y_valid))
# h4 = model_test_4.fit(X_train, y_train, epochs=5,
#                  validation_data=(X_valid, y_valid))
# h5 = model_test_5.fit(X_train, y_train, epochs=5,
#                  validation_data=(X_valid, y_valid))
# h6 = model_test_6.fit(X_train, y_train, epochs=5,
#                  validation_data=(X_valid, y_valid))

In [9]:
# import matplotlib.pyplot as plt

In [10]:
# h = [h1, h2, h3, h4, h5, h6]
# for i, h_i in enumerate(h):
#     plt.plot(h_i.history['loss'], label=f"Training loss 1e-{i+1}")
#     plt.legend()

In [11]:
import scikeras
from scikeras.wrappers import KerasRegressor

In [12]:
es = tf.keras.callbacks.EarlyStopping(patience=10, min_delta=1.0, verbose=1)

In [13]:
keras_reg = KerasRegressor(build_model, callbacks=[es])

In [14]:
from sklearn.model_selection import RandomizedSearchCV

In [15]:
rnd_search_cv = RandomizedSearchCV(keras_reg, param_distribs,
                                   n_iter=5,
                                   cv=3,
                                   verbose=2)

rnd_search_cv.fit(X_train, y_train, epochs=100, validation_data=(X_valid,y_valid), verbose=0)

Fitting 3 folds for each of 5 candidates, totalling 15 fits


2025-05-23 21:55:09.229206: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4 Pro
2025-05-23 21:55:09.229288: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 48.00 GB
2025-05-23 21:55:09.229326: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 18.00 GB
I0000 00:00:1748030109.229350 4641354 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1748030109.229389 4641354 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)
2025-05-23 21:55:09.457075: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


Epoch 11: early stopping
121/121 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
[CV] END model__learning_rate=0.0017605217881778957, model__n_hidden=3, model__n_neurons=8, model__optimizer=sgd; total time=  13.4s
Epoch 11: early stopping
121/121 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
[CV] END model__learning_rate=0.0017605217881778957, model__n_hidden=3, model__n_neurons=8, model__optimizer=sgd; total time=  13.0s
Epoch 12: early stopping
121/121 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  
[CV] END model__learning_rate=0.0017605217881778957, model__n_hidden=3, model__n_neurons=8, model__optimizer=sgd; total time=  13.5s
Epoch 12: early stopping
121/121 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
[CV] END model__learning_rate=0.0008971518379967991, model__n_hidden=1, model__n_neurons=50, model__optimizer=sgd; total time=  12.8s
Epoch 11: early stopping
121/121 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  
[CV] END model__learning_rate=0.0008971518379967991, model__n_hidden=1, model__n_neurons=50, model__optimizer=sgd; total time=  12.0s

RandomizedSearchCV(cv=3,
                   estimator=KerasRegressor(callbacks=[<keras.src.callbacks.early_stopping.EarlyStopping object at 0x32e62e490>], model=<function build_model at 0x3317ccea0>),
                   n_iter=5,
                   param_distributions={'model__learning_rate': [0.002159444633801102,
                                                                 0.0005627298711069277,
                                                                 0.002566441850953288,
                                                                 0.002974339856844315,
                                                                 0.0017597097369899564,
                                                                 0.00033938433419...
                                                                 0.011579855441592722,
                                                                 0.0012636462309306754,
                                                                 0.0008943423884549597,
                                                                 0.0005789492098221247,
                                                                 0.0005680875627960411,
                                                                 0.0010336391903572118,
                                                                 0.0041556667223365895,
                                                                 0.001779298247734286, ...],
                                        'model__n_hidden': [0, 1, 2, 3],
                                        'model__n_neurons': [1, 2, 3, 4, 5, 6,
                                                             7, 8, 9, 10, 11,
                                                             12, 13, 14, 15, 16,
                                                             17, 18, 19, 20, 21,
                                                             22, 23, 24, 25, 26,
                                                             27, 28, 29, 30, ...],
                                        'model__optimizer': ['adam', 'sgd',
                                                             'nesterov']},
                   verbose=2)

In [16]:
rnd_search_cv.best_params_

{'model__optimizer': 'sgd',
 'model__n_neurons': 50,
 'model__n_hidden': 1,
 'model__learning_rate': 0.0008971518379967991}

In [17]:
import pickle

with open('rnd_search_params.pkl', 'wb') as f:
    pickle.dump(rnd_search_cv.best_params_, f)

with open('rnd_search_scikeras.pkl', 'wb') as f:
    pickle.dump(rnd_search_cv, f)

# Zadanie 3 - Przeszukiwanie przestrzeni hiperparametrów przy pomocy Keras Tuner

In [18]:
import keras_tuner as kt

def build_model_kt(hp):
    n_hidden = hp.Int("n_hidden", min_value=0, max_value=3, default=2)
    n_neurons = hp.Int("n_neurons", min_value=1, max_value=100)
    learning_rate = hp.Float("learning_rate", min_value=3e-4, max_value=3e-2, sampling="log")
    optimizer = hp.Choice("optimizer", values=["sgd", "adam", "nesterov"])
    
    if optimizer == "sgd":
        optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate)
    elif optimizer == "nesterov":
        optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate, nesterov=True)
    else:
        optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
        
    model = tf.keras.Sequential()
    model.add(tf.keras.layers.Input(shape=[8]))
    
    for _ in range(n_hidden):
        model.add(tf.keras.layers.Dense(n_neurons, activation="relu"))
    
    model.add(tf.keras.layers.Dense(1))
    
    model.compile(loss="mse", optimizer=optimizer, metrics=["mse"])
    
    return model

In [19]:
random_search_tuner = kt.RandomSearch(build_model_kt, objective="val_loss", max_trials=10,
                                      overwrite=True, directory="my_california_housing",
                                      project_name="my_rnd_search", seed=42)

In [20]:
import os
root_logdir = os.path.join(random_search_tuner.project_dir, 'tensorboard')
tb = tf.keras.callbacks.TensorBoard(root_logdir)

In [21]:
random_search_tuner.search(X_train, y_train, epochs=100,
                           validation_data=(X_valid, y_valid),
                           callbacks=[es, tb])

Trial 10 Complete [00h 00m 42s]
val_loss: 1.7623149156570435

Best val_loss So Far: 0.5159376859664917
Total elapsed time: 00h 05m 49s


In [22]:
best_model = random_search_tuner.get_best_models(1)[0]

In [23]:
best_hps = random_search_tuner.get_best_hyperparameters(num_trials=1)[0]
best_hps.values

{'n_hidden': 3,
 'n_neurons': 46,
 'learning_rate': 0.001652854166593675,
 'optimizer': 'nesterov'}

In [24]:
with open('kt_search_params.pkl', 'wb') as f:
    pickle.dump(best_hps.values, f)

In [25]:
best_model.save('kt_best_model.keras')